# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to explore and process a Croissant-structured dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) package, based on the FAIR^2 dataset package schema.

### Dataset Source
Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```


In [ ]:
# Install mlcroissant if not present
!pip install --quiet mlcroissant

## 1. Data Loading
Load the dataset metadata and discover available record sets with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset title and description
print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
This section prints all available record sets, their `@id`s, and lists the fields for each. 

> **All entities (record sets, fields, etc.) are referenced by their `@id` values.**

In [ ]:
# List all record sets by their `@id` and list their fields' `@id`s
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in the Croissant schema.")
else:
    for rs in record_sets:
        print(f"Record Set: {rs['@id']}")
        if 'field' in rs:
            fields = rs['field']
            if isinstance(fields, dict):
                # Single field case
                fields = [fields]
            field_ids = [f['@id'] if isinstance(f, dict) and '@id' in f else f for f in fields]
        else:
            field_ids = []
        print("  Fields:")
        for field_id in field_ids:
            print(f"    - {field_id}")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s as discovered above.

In [ ]:
# Identify record set(s) to extract; we'll extract all found as examples
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for rs_id in record_set_ids:
    print(f"Extracting records from record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"  Loaded {len(records)} records.")
    else:
        print("  No records found.")
    print()
# If any non-empty DataFrame exists, print its columns and first rows for inspection
if dataframes:
    first_rs_id = next(iter(dataframes))
    print(f"Columns in {first_rs_id}:\n", dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No tabular data extracted from record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing: filtering, normalization, and grouping using field `@id` references. 
Edit the example below as needed to suit the record set and fields loaded above.

In [ ]:
# EDA Example based on first DataFrame loaded
if dataframes:
    # Pick the first available DataFrame and try to identify numeric fields
    rs_id = first_rs_id
    df = dataframes[rs_id]
    numeric_candidates = df.select_dtypes('number').columns.tolist()
    print(f"Numeric fields in {rs_id}: {numeric_candidates}")
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Analyzing field by @id: {numeric_field}")

        threshold = df[numeric_field].mean() if not pd.isnull(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Attempt grouping by another field if available
        non_numeric_candidates = [col for col in df.columns if col != numeric_field]
        group_field = None
        for col in non_numeric_candidates:
            if df[col].nunique() < min(10, len(df)/2):  # Only group if few unique values
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped by {group_field} (mean of {numeric_field}):")
            display(grouped_df.head())
        else:
            print("No suitable non-numeric field found for grouping.")
    else:
        print(f"No numeric fields found in record set {rs_id} for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Plot a distribution and relationship from filtered data (if any) using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_candidates:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field} (by @id)')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field} (by @id)')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and process tabular data from a Croissant-powered FAIR^2 dataset using `mlcroissant`. We explored `recordSet` and `field` `@id`s, loaded data into DataFrames, and applied exploratory analysis and visualization. 

**Remember:** All references are made via entity `@id` fields for full schema traceability and reproducibility. Continue your analysis or modeling using this structure as a template for the broader dataset!